In [1]:
import scipy.sparse as sp
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import time
import torch
import torch.nn.functional as F
import torch.optim as optim
import torch.nn as nn
import scvelo as scv
from sklearn.decomposition import PCA
#from utils import seed_everything, load_data, sigmoid_gumbel_sample, get_logger, makedirs, count_parameters, predict_trajectory
from parse import parser
#from model import build_model_tabular, set_cnf_options,build_model_spatial
from lib.regularization import create_regularization_fns
from train import pred_train, graph_discov
from visualize import plot_output, plot_causal_matrix
import os
from sklearn.neighbors import kneighbors_graph

In [2]:
import numpy as np
import spateo as spt
import torch
import scanpy as sc
import scvelo as scv
import anndata as ad
import pandas as pd

In [3]:
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from matplotlib.pyplot import rc_context
from matplotlib.colors import Normalize, TwoSlopeNorm
from matplotlib.cm import ScalarMappable
import matplotlib.ticker as ticker

In [4]:
import numpy as np
import spateo as spt
import torch
import scanpy as sc
import scvelo as scv
import anndata as ad
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from matplotlib.pyplot import rc_context
from matplotlib.colors import Normalize, TwoSlopeNorm
from matplotlib.cm import ScalarMappable
import matplotlib.ticker as ticker
import scipy.sparse as sp
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import time
import torch
import torch.nn.functional as F
import torch.optim as optim
import torch.nn as nn
import scvelo as scv
from sklearn.decomposition import PCA
from utils import seed_everything, load_data, sigmoid_gumbel_sample, get_logger, makedirs, count_parameters
from parse import parser
from model import build_model_tabular, set_cnf_options
from lib.regularization import create_regularization_fns
from train import pred_train, graph_discov
from visualize import plot_output, plot_causal_matrix
import os
from sklearn.neighbors import kneighbors_graph
import numpy as np
import torch
import random

In [ ]:
adata= spt.read_h5ad('./sim_datset/sim_processed_adata_with_scaler.h5ad')
adata

AnnData object with n_obs × n_vars = 4138 × 3
    obs: 'time', 'true_growth_rate', 'true_death_rate', 'true_net_growth', 'time_point'
    uns: 'latent_scaler_mean', 'latent_scaler_std'
    obsm: 'spatial', 'spatial_aligned_gt'

In [ ]:
import sys, importlib
from layers import CausalMix  
exp_model_path = './sim_datset/sim_graph_checkpt.pth'
#spatial_path = '/media/lenovo/A06B2FA1620B6FCB/pythonProject/models/Rspatial_checkpt.pth'

model = torch.load(exp_model_path, map_location=torch.device('cpu'))
#spatial_model = torch.load(spatial_path, map_location=torch.device('cpu'))


has CausalMix: True


In [ ]:
class GrowthNet(torch.nn.Module):
    def __init__(self):
        super().__init__()

        self.fc1 = torch.nn.Linear(4, 64)
        self.fc2 = torch.nn.Linear(64, 64)
        self.fc3 = torch.nn.Linear(64, 1)

    def forward(self, x):
        x = torch.nn.functional.leaky_relu(self.fc1(x))
        x = torch.nn.functional.leaky_relu(self.fc2(x))
        x = self.fc3(x)
        return x
growth_model_path= './sim_datset/grow_model_1.0.pth'
growth_model = torch.load(growth_model_path, map_location=torch.device('cpu'))


In [8]:
integral_time = list(np.unique(adata.obs['time_point']))
time_pts = range(len(integral_time))
integral_time


[0, 1, 2, 3, 4, 5]

In [ ]:
def raw_gene_space_growth(growth_model_path, device=torch.device('cpu')):
    
    growth_model = torch.load(growth_model_path, map_location=device)
    growth_model.eval()
    
    def growth(encoded_gene, t):
       
        if not isinstance(encoded_gene, torch.Tensor):
            encoded_gene = torch.tensor(encoded_gene, dtype=torch.float32).to(device)
        
        
        batch_size = encoded_gene.shape[0]
        time_tensor = torch.tensor(t, dtype=torch.float32).repeat(batch_size, 1).to(device)
        
        input_data = torch.cat([encoded_gene, time_tensor], axis=1)
        
       
        with torch.no_grad():
            cell_growth = growth_model(input_data)
        #g_min = torch.min(cell_growth)
        #g_max = torch.max(cell_growth)
        #cell_growth_norm = 2 * (cell_growth - g_min) / (g_max - g_min) - 1
        return cell_growth
    
    return growth

In [21]:
growth_gene = raw_gene_space_growth(growth_model_path, device=torch.device('cpu'))
#growth_gene
#velocity_T = np.zeros_like(adata.X)
#velocity_ae = np.zeros_like(adata.obsm['X_latent'])
#velocity_spatial = np.zeros_like(adata.obsm['spatial_aligned'])
cell_growth = np.zeros((adata.n_obs, 1))
#cell_growth
# compute cell RNA velocity, spatial velocity and growth


In [22]:
for i in time_pts:
    cur_cell_gene = torch.tensor(adata[adata.obs.time_point==integral_time[i], :].X, requires_grad=False, dtype=torch.float32)
    #cur_cell_spatial = torch.tensor(adata[adata.obs.time==integral_time[i], :].obsm['X_spatial'], requires_grad=False, dtype=torch.float32)

    
    cur_cell_growth = growth_gene(cur_cell_gene, integral_time[i])
    
    
    cell_growth[adata.obs.time_point==integral_time[i]] = cur_cell_growth.detach().numpy()


In [23]:
adata.obs['cell_growth'] = cell_growth.squeeze()
var_names = adata.var_names
var_names_new = [s.split("|")[0].strip() for s in list(var_names)]
adata.var_names = var_names_new
adata.var['var_names_new'] = var_names_new


In [15]:
adata

AnnData object with n_obs × n_vars = 4138 × 3
    obs: 'time', 'true_growth_rate', 'true_death_rate', 'true_net_growth', 'time_point', 'cell_growth', 'time_batch'
    var: 'var_names_new'
    uns: 'time_batch_colors'
    obsm: 'spatial', 'spatial_aligned_gt'

In [24]:
adata.obs['time_batch'] = ['']*adata.n_obs
adata.obs['time_batch'][adata.obs.time_point==integral_time[0]] = 0
adata.obs['time_batch'][adata.obs.time_point==integral_time[1]] = 1
adata.obs['time_batch'][adata.obs.time_point==integral_time[2]] = 2
adata.obs['time_batch'][adata.obs.time_point==integral_time[3]] = 3
adata.obs['time_batch'][adata.obs.time_point==integral_time[4]] = 4
adata.obs['time_batch'][adata.obs.time_point==integral_time[5]] = 5
adata.uns['time_batch_colors'] = ['#E74C3C', '#3498DB', '#F1C40F', '#2ECC71', '#9B59B6', "#103BE5"]


In [25]:
adata_0 = adata[adata.obs.time_point==integral_time[0], :]
adata_1 = adata[adata.obs.time_point==integral_time[1], :]
adata_2 = adata[adata.obs.time_point==integral_time[2], :]
adata_3 = adata[adata.obs.time_point==integral_time[3], :]
adata_4 = adata[adata.obs.time_point==integral_time[4], :]
adata_5 = adata[adata.obs.time_point==integral_time[5], :]


In [15]:
growth_true = adata_5.obs['true_net_growth'].values 
growth_pred = adata_5.obs['cell_growth'].values

In [ ]:
save_path = './sim_datset'
## Plot the aligned data
# Set parameters
sc.set_figure_params(dpi=150, dpi_save=400, frameon=False, fontsize=8, transparent=True)
sc.settings.verbosity = 0
sc.logging.print_header()


scanpy==1.9.8 anndata==0.9.2 umap==0.5.7 numpy==1.22.4 scipy==1.9.3 pandas==1.5.3 scikit-learn==1.3.2 statsmodels==0.14.1 igraph==0.11.8 pynndescent==0.5.13


In [27]:
adata.obs['cell_growth'] = cell_growth.squeeze()


In [ ]:
def load_data_from_anndata(adata, time_key='time_point', spatial_key='spatial_aligned_gt'):
   
    
    train_time = adata.obs[time_key].unique()
    
    train_time = np.sort(train_time)
    
    
    data_train = {}
    for t in train_time:
        
        adata_t = adata[adata.obs[time_key] == t]
        
    
        
        gene_expr = adata_t.X
        
       
        
        spatial_coords = adata_t.obsm[spatial_key]
       
        cell_data = np.concatenate([gene_expr, spatial_coords], axis=1)
        
        
        data_train[t] = torch.tensor(cell_data, dtype=torch.float32)
    
    return data_train,train_time


data_train,train_time = load_data_from_anndata(adata)
init_cell_5DPI = data_train[train_time[1]]


In [ ]:
cnf_model_path = "./sim_datset/sim_cnf_spatial.pth" 
from cnf_OT2 import SpatialMorphogenesisGenerator
device = 'cuda' if torch.cuda.is_available() else 'cpu'
time_series_data = [adata[adata.obs['time_point'] == tp] for tp in [0,1,2,3,4,5] if len(adata[adata.obs['time_point'] == tp])>0]
cnf_generator = SpatialMorphogenesisGenerator.load(
        load_path=cnf_model_path,
        time_series_data=time_series_data,
       #growth_model=growth_model,
        device=device
    )
cnf_generator.cnf.eval()


Preprocessing spatial and expression data...
Computing OT pairs...
Model loaded from: /mnt/sda2/causal_1105/results/tmp/embry2/sim_spatial_ode.pth


ContinuousTimeCNF(
  (odefunc): ContinuousTimeODEfunc(
    (diffeq): SpatialODENet(
      (net): Sequential(
        (0): Linear(in_features=6, out_features=256, bias=True)
        (1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (2): Tanh()
        (3): Linear(in_features=256, out_features=256, bias=True)
        (4): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (5): Tanh()
        (6): Linear(in_features=256, out_features=256, bias=True)
        (7): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (8): Tanh()
        (9): Linear(in_features=256, out_features=2, bias=True)
      )
    )
  )
)

In [ ]:
#from sklearn.preprocessing import StandardScaler
from cnf_OT2 import disperse_points
import torch
import numpy as np
from sklearn.preprocessing import StandardScaler

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

def evolution_forward(init_cell, model, growth_model, init_time, end_time, adata,
                      delta_t=0.2, trained_scaler=None, max_cells=20000,
                      cnf_generator=None):
    
    device = next(model.parameters()).device
    print(f"Model on: {device}")
    
    
    scaler = StandardScaler()
    scaler.mean_ = adata.uns['latent_scaler_mean']
    scaler.scale_ = adata.uns['latent_scaler_std']
    scaler.n_features_in_ = len(scaler.mean_)

    def growth(x, delta_t, growth_model, t, cur_cell_id, sigma_d=0.00001):
        device = x.device 
        x_np = x.detach().cpu().numpy()
        y_np = x_np[:, 0:-2] 
        coords_np = x_np[:, -2:] 
        
        y_transformed = torch.tensor(y_np, dtype=torch.float32, device=device)
        coords = torch.tensor(coords_np, dtype=torch.float32, device=device)
        
        cell_number = x.shape[0]
        x_growth = []
        y_growth = []
        new_cell_id = []
        
        if not isinstance(t, (int, float)):
            t = t.item() 
        t_tensor = torch.tensor(t, dtype=torch.float32, device=device).view(1, 1).repeat(cell_number, 1)
        
        combined_input = torch.cat([y_transformed, t_tensor], axis=1)
        g = growth_model(combined_input)
        for i in range(cell_number):
            g_i = g[i, :] 
            temp = np.random.rand()
           
            if g_i > 1.0 and temp < (g_i * delta_t):
                x_growth.append(y_transformed[i,:])
                y_growth.append(coords[i,:])
                new_cell_id.append(cur_cell_id[i]) 
                
                new_gene = y_transformed[i,:] + sigma_d * torch.randn_like(y_transformed[i,:])
                new_coords= coords[i,:] + sigma_d * torch.randn_like(coords[i,:])
                angle = np.random.uniform(0, 2 * np.pi)
                distance = np.random.exponential(1) + 0.1
                offset_np = np.array([np.cos(angle), np.sin(angle)]) * distance
                offset = torch.tensor(offset_np, dtype=torch.float32, device=device)
                new_coord = new_coords + offset
                x_growth.append(new_gene)
                y_growth.append(new_coord)
                new_cell_id.append(f"{cur_cell_id[i]}_child") 
            
            elif g_i > 1.0 and temp > (g_i * delta_t):
                x_growth.append(y_transformed[i,:])
                y_growth.append(coords[i,:])
                new_cell_id.append(cur_cell_id[i])
            
            elif g_i < 1.0 and temp > (-g_i * delta_t):
                x_growth.append(y_transformed[i,:])
                y_growth.append(coords[i,:])
                new_cell_id.append(cur_cell_id[i])
        
        if len(x_growth) > max_cells:
            keep_idx = np.random.choice(len(x_growth), max_cells, replace=False)
            x_growth = [x_growth[idx] for idx in keep_idx]
            y_growth = [y_growth[idx] for idx in keep_idx] 
            new_cell_id = [new_cell_id[idx] for idx in keep_idx]
        
        if not x_growth:
            return x, cur_cell_id
        
        cur_gene = torch.vstack(x_growth).type(torch.float32)
        cur_coord = torch.vstack(y_growth).type(torch.float32)
        cur_cell = torch.cat([cur_gene, cur_coord], dim=1)
        
        return cur_cell, new_cell_id
    
    # ---------------------- main ----------------------
    int_tps = np.linspace(init_time, end_time, int((end_time - init_time)/delta_t) + 1)
    int_tps = np.round(int_tps, 1)
    ntimes = len(int_tps)
   
    all_exp = []
    all_spa = []
    all_cell_type = []
    all_cell_id = []
    
    cur_cell = init_cell.to(device, dtype=torch.float32)
    
    cur_cell_id = [f"cell_{i}" for i in range(cur_cell.shape[0])]
    
    for i in range(ntimes):
        all_exp.append(cur_cell[:, 0:-2].detach().cpu().numpy())
        all_spa.append(cur_cell[:, -2:].detach().cpu().numpy())
        all_cell_id.append(cur_cell_id)
        
        if i >= ntimes - 1:
            break
        
        current_time = int_tps[i]
        next_time = int_tps[i+1]
        time_step = next_time - current_time
        
        
        cur_cell, cur_cell_id = growth(
            x=cur_cell,
            delta_t=time_step,
            growth_model=growth_model,
            t=current_time,
            cur_cell_id=cur_cell_id,
            sigma_d=0.00001
        )
        
       
      
        z_gene = cur_cell[:, 0:-2].detach().cpu().numpy()
        z_gene_scaled = scaler.transform(z_gene)
        gene_scaled_tensor = torch.from_numpy(z_gene_scaled).to(device, dtype=torch.float32)
        cnf = model.chain[0]
        zero = torch.zeros(gene_scaled_tensor.shape[0], 1, device=device)
        
        integration_times = torch.tensor([current_time, next_time], dtype=torch.float32, device=device)
        gene_output, _ = cnf(gene_scaled_tensor, zero, integration_times=integration_times, reverse=False)
        gene_output_np = gene_output.detach().cpu().numpy()
        gene_output_final = scaler.inverse_transform(gene_output_np)
        
        cur_cell[:, 0:-2] = torch.from_numpy(gene_output_final).to(device, dtype=torch.float32) 
        
       
        z_spatial = cur_cell[:, -2:].detach().cpu().numpy() 
        z_spatial_relative = cnf_generator.coord_processor.transform(z_spatial, current_time)
        z_spatial_tensor = torch.tensor(z_spatial_relative, dtype=torch.float32, device=device)
        
        
        cur_gene_tensor = cur_cell[:, 0:-2].to(device, dtype=torch.float32)
        
       
        if cur_gene_tensor.shape[1] != latent_dim:
            if cur_gene_tensor.shape[1] > latent_dim:
                cur_gene_tensor = cur_gene_tensor[:, :latent_dim]
            else:
                pad = torch.zeros((cur_gene_tensor.shape[0], latent_dim - cur_gene_tensor.shape[1]), device=device)
                cur_gene_tensor = torch.cat([cur_gene_tensor, pad], dim=1)
        
        
        state_input = torch.cat([z_spatial_tensor, cur_gene_tensor], dim=1)
        
        time_diff = next_time - current_time  
        times = torch.tensor([0.0, time_diff], dtype=torch.float32, device=device)
        
        with torch.no_grad():
            
            predicted_full = cnf_generator.cnf(state_input, times)
            
         
            predicted_relative = predicted_full[:, :2]
            
        predicted_relative_np = predicted_relative.cpu().numpy()
        
        
        predicted_relative_dispersed = disperse_points(predicted_relative_np, min_dist=0.01, max_iter=100)
        
        
        predicted_coords = cnf_generator.coord_processor.inverse_transform(predicted_relative_dispersed, next_time)
      
        cur_cell[:, -2:] = torch.tensor(predicted_coords, dtype=cur_cell.dtype, device=device)
        
    return all_exp, all_spa, all_cell_id, int_tps

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


model = model.to(device)
growth_model = growth_model.to(device)
#cnf_generator = cnf_generator.to(device)  

cnf_generator.cnf = cnf_generator.cnf.to(device)
#cnf_generator.expr_model = cnf_generator.expr_model.to(device)
#cnf_generator.growth_model = cnf_generator.growth_model.to(device)
cnf_generator.ode_net = cnf_generator.ode_net.to(device)


init_cell_5DPI = init_cell_5DPI.to(device)

In [ ]:
import anndata as ad
import pandas as pd
import numpy as np


# from 2DPI to 3.5DPI
init_cell_2DPI = data_train[train_time[4]]
growth_model.eval()  

common_initial_time_point = 0
init_time =4- common_initial_time_point
end_time = 5 #- common_initial_time_point


all_exp, all_spa, all_cell_id, all_time_point = evolution_forward(
    init_cell=init_cell_2DPI,
    model=model,
    growth_model=growth_model, 
     adata=adata, 
    init_time=init_time,
    end_time=end_time,
    delta_t=0.5,cnf_generator=cnf_generator
)
final_gene_expr = all_exp[-1]  
final_cell_id = all_cell_id[-1]


adata_St54 = ad.AnnData(
    X=final_gene_expr,  
    obs=pd.DataFrame(index=final_cell_id)  
)

final_spatial = all_spa[-1]
adata_St54.obsm['spatial'] = final_spatial 
# 保存为h5ad文件
#adata_St54.write_h5ad("predicted_final_timepoint1.h5ad")
#print("AnnData文件已保存为 predicted_final_timepoint1.h5ad")

# 绘图：若plot_after_alignment_scanpy依赖细胞类型，需调整（比如只画空间分布/基因表达）
# 注：若绘图函数必须要细胞类型，需注释/修改该函数
# plot_after_alignment_scanpy(adata_St54, figsize=(6, (5*2/3)), pointsize=8, title='St49', show_legend=False,save_path='/media/lenovo/A06B2FA1620B6FCB/pythonProject/results/tmp/embry2/mouse/St49.pdf')

Model on: cuda:0
正向递推时间点：[4.  4.5 5. ]（共3个）
初始细胞形状: torch.Size([765, 5]), 设备: cuda:0
细胞形状: torch.Size([747, 5]), 设备: cuda:0
细胞形状: torch.Size([740, 5]), 设备: cuda:0
保存的AnnData形状：(740, 3)
最后一个时间点的细胞数量：740


In [71]:
import numpy as np
import ot
from scipy.spatial.distance import cdist

def w2_distance_gene_and_space_with_error(
    X_pred_gene: np.ndarray,
    X_pred_space: np.ndarray,
    X_true_gene: np.ndarray,
    X_true_space: np.ndarray,
    w: float = 0.9,
    sample_size: int = 10000,
    n_repeats: int = 5,
) -> tuple:
    """
    联合空间（基因+空间）的 W2 距离，并返回多次子采样的均值和标准差。
    """
    n = X_pred_gene.shape[0]
    m = X_true_gene.shape[0]
    w2_list = []

    for _ in range(n_repeats):
        # 随机子采样
        if n > sample_size:
            idx_pred = np.random.choice(n, sample_size, replace=False)
        else:
            idx_pred = np.arange(n)
        if m > sample_size:
            idx_true = np.random.choice(m, sample_size, replace=False)
        else:
            idx_true = np.arange(m)

        X_pred_gene_sub = X_pred_gene[idx_pred]
        X_pred_space_sub = X_pred_space[idx_pred]
        X_true_gene_sub = X_true_gene[idx_true]
        X_true_space_sub = X_true_space[idx_true]

        a = np.ones(X_pred_gene_sub.shape[0]) / X_pred_gene_sub.shape[0]
        b = np.ones(X_true_gene_sub.shape[0]) / X_true_gene_sub.shape[0]

        M_gene = cdist(X_pred_gene_sub, X_true_gene_sub, metric="sqeuclidean")
        M_space = cdist(X_pred_space_sub, X_true_space_sub, metric="sqeuclidean")
        M = w * M_gene + (1 - w) * M_space

        W2_sq = ot.emd2(a, b, M)   # 精确 W2
        w2_list.append(np.sqrt(W2_sq))

    return np.mean(w2_list), np.std(w2_list)
#scale_factor = adata.uns.get('spatial_scale_factor')

mean_w2, std_w2 = w2_distance_gene_and_space_with_error(
    X_pred_gene=adata_St54.X,
    X_pred_space=adata_St54.obsm['spatial'] ,
    X_true_gene=adata_2.X,
    X_true_space=adata_2.obsm['spatial'] ,
    w=0.9,
    sample_size=594,
    n_repeats=5
)
print(f"W2 = {mean_w2:.6f} ± {std_w2:.6f}")

W2 = 0.789769 ± 0.014523


In [ ]:

%matplotlib inline  
import matplotlib.pyplot as plt
import numpy as np
import anndata as ad  

plt.figure(figsize=(5, 4))  
target_time = 2.25
subset = adata_St54


coords = subset.obsm['spatial']
genes = subset.X


r_val = genes[:, 0] / (np.percentile(genes[:, 0], 99) + 0.01)
b_val = genes[:, 1] / (np.percentile(genes[:, 1], 99) + 0.01)
r_val = np.clip(r_val, 0, 1)
b_val = np.clip(b_val, 0, 1)

colors = np.zeros((len(coords), 3))
colors[:, 0] = r_val  
colors[:, 2] = b_val  


plt.scatter(
    coords[:, 0], coords[:, 1], 
    c=colors, 
    s=70,          
    alpha=0.5,     
    edgecolors='none'  
)


plt.title(f"Time: {target_time}s (N={len(coords)})", fontsize=16, fontweight='bold')
plt.axis('off')  
plt.gca().set_aspect('equal')  
plt.tight_layout()  
plt.savefig(
    "./sim_datset/pred_t.pdf", 
    dpi=300,                         
    bbox_inches='tight',               
    pad_inches=0.1,                    
    format='pdf'                       #
)

plt.show()  

